# Eksperimen 2 — CAD-ViT V2 Ben Graham σ=10

**Pertanyaan penelitian:** bagaimana hasil kandidat saat preprocessing diberi enhancement BG10 dan protokol training V2? [Ringkasan utama](../../CAD_ViT.ipynb) · [V1](01_experiment_initial_cadvit_v1.ipynb) · [arsip historis](../archive/CAD_ViT_original_105_cells.ipynb). Ini eksperimen historis; V2 **65/30 epoch** dan V2 **80/80 epoch** adalah run berbeda.

## Identitas, dataset, dan protokol

| Seri | APTOS2019 | DDR | Bukti yang tersedia |
| --- | --- | --- | --- |
| V2 awal | `aptos_v2_bg10_stable_65ep_seed42` (65 epoch) | `ddr_v2_bg10_stable_30ep_seed42` (30 epoch) | Ringkasan validation dari log luar repo di arsip §5.8; final test tidak tersedia lokal. |
| V2 80 epoch | `aptos_v2_bg10_stable_80ep_seed42` (80 epoch) | `ddr_v2_fp32_weighted_fp64clip_seed42` (80 epoch campuran) | Tabel epoch di bawah; dua JSON final test lokal berhash cocok dengan catatan server. |

APTOS menggunakan 3.662 pasangan label–gambar; DDR grading menggunakan 12.522 label grade 0–4. Split reproduksi 8:1:1, seed 42, dipertahankan menurut kode historis; final test berisi 366 dan 1.252 gambar. JSON V2 mencatat manifest SHA-256 APTOS `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` dan DDR `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1`; isi manifest tidak ada lokal.

V2 mempertahankan crop/resize/Gaussian V1 lalu menambahkan Ben Graham RGB float32: `clip(4I − 4GaussianBlur(I, σ=10) + 128, 0, 255)`. Ini **extension eksperimen**, bukan langkah yang eksplisit dari paper CAD-ViT. Train memakai augmentasi flip/shear/rotation, shuffle biasa tanpa sampler berbobot pada fase awal. Target LR 0,000186, batch fisik/efektif 16, seed 42, 80 epoch; AdamW, weight decay 0, cross-entropy, AMP+GradScaler bila CUDA, grad clip norm 1, tanpa scheduler/early stopping, dan checkpoint berdasarkan validation macro-F1 adalah asumsi implementasi. DDR 80 epoch berubah objective/precision setelah epoch 59, jadi deskripsi awal ini tidak berlaku homogen untuk semua epoch.

## Walkthrough kode dan perbedaan dari V1

| Langkah | Source | Input → output; perubahan V2 |
| --- | --- | --- |
| BG10 | [Arsip sel 41: `preprocess_fundus_v2`](../archive/CAD_ViT_original_105_cells.ipynb), reusable [`preprocessing.py`](../cadvit/preprocessing.py) | Gambar RGB setelah jalur V1 → uint8 BG10; σ=10 dan koefisien merupakan konfigurasi historis. |
| Data/split | Arsip sel 45/47: `prepare_manifest`, `FundusDataset`, `make_loaders` | Manifest → batch; split V1 diwarisi, V2 tanpa `WeightedRandomSampler` awal. |
| Model | Arsip sel 53/55, reusable [`model.py`](../cadvit/model.py) | Tensor → lima logit; kandidat arsitektur sama, tidak mengklaim bobot/checkpoint sama. |
| Runner V2 | Arsip sel 68: `v2_experiment_config`, `score_loader_v2`, `_train_epoch_v2`, `run_experiment_v2` | Config/manifest → history/checkpoint; guard identitas run dan pemeriksaan non-finite. Ini bukan runner modular [`experiments.py`](../cadvit/experiments.py). |
| Evaluasi V2 80 epoch | [`evaluation.py`](../cadvit/evaluation.py): `preflight`, `classification_metrics`, `run_final_test` | Checkpoint/manifest beku → JSON agregat. Dokumen [preflight historis](../../docs/V2_FINAL_EVALUATION.md) mencatat guard; jangan jalankan final test ulang. |

Source log dan checkpoint V2 asli tidak ada lokal. Kesesuaian source evaluator dengan proses server tidak dibuktikan oleh hash JSON hasil saja. `model.py`/`preprocessing.py` cocok secara AST dengan definisi historis terkait, tetapi itu bukan bukti versi runtime server.

## Fase V2 awal: 65/30 epoch

| Dataset/run | Epoch tercatat | Best validation epoch | Validation macro-F1 | Status final test |
| --- | ---: | ---: | ---: | --- |
| APTOS `aptos_v2_bg10_stable_65ep_seed42` | 1–65 | 65 | 0,650653821679372 | Tidak tersedia lokal untuk run ini |
| DDR `ddr_v2_bg10_stable_30ep_seed42` | 1–30 | 24 | 0,5877264724487317 | Tidak tersedia lokal untuk run ini |

Log mentah seri ini berada di luar repo; ringkasan historisnya ada di [arsip §5.8](../archive/CAD_ViT_original_105_cells.ipynb). Karena 65/30 baris sumber tidak tersedia lokal, tabel lengkap atau kurva baru tidak dibuat. Skor ini tidak digabung dengan seri 80/80 atau final testnya.

## Fase V2 80/80 epoch dan recovery DDR

APTOS 80 epoch memakai BG10/AMP. DDR: **epoch 1–59** AMP FP16 + unweighted CE; **60–63** weighted FP32; **64–80** weighted FP32 dengan norm gradien dihitung FP64. Weighted loss dan FP64 clip bukan metode yang ditentukan paper. Log dan source mendukung urutan itu, sedangkan transfer checkpoint epoch 63 belum diverifikasi dari biner/transkrip lokal. Loss sebelum/sesudah perubahan objective tidak dapat dibandingkan sebagai satu fungsi yang sama. Seleksi checkpoint tetap dari validation macro-F1.

Sumber log server yang dicatat di [arsip §5.5](../archive/CAD_ViT_original_105_cells.ipynb): APTOS `c81a2d3b3bfafe03ecb390bce2e2839d3d1bd7a4db0575294a7e6fd1b5b4ed08`; DDR fase AMP `4101e32a0028211ad3f0a0126050b5c0e3c2ce2d7d01edb699b241856b5ca24b`, weighted FP32 `379f1a53ee2871c5eacdda1f89f980094e35bfd871a5cbe6027c9a72b6eef9b6`, FP64 clip `2ec854cd4eccacd53f595e2ffc71347ebdc2a344e2bbc984fe85247fd1518932`. Hash log itu belum dapat dihitung ulang lokal.

### APTOS2019 V2 BG10 — 80 epoch validation

| Epoch | Train Loss | Validation Loss | Validation Accuracy | Validation Macro-F1 |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 0.927561940 | 1.207536785 | 0.663934426 | 0.297769688 |
| 2 | 0.779322600 | 0.947620418 | 0.644808743 | 0.318204834 |
| 3 | 0.741657620 | 0.765772699 | 0.699453552 | 0.386034730 |
| 4 | 0.705662248 | 0.668784613 | 0.751366120 | 0.450302402 |
| 5 | 0.668807328 | 0.789421816 | 0.715846995 | 0.400761384 |
| 6 | 0.657152622 | 0.627541132 | 0.754098361 | 0.501868648 |
| 7 | 0.635783534 | 0.635500810 | 0.754098361 | 0.467216536 |
| 8 | 0.606194885 | 0.981338208 | 0.663934426 | 0.377414972 |
| 9 | 0.591275607 | 0.966783020 | 0.680327869 | 0.448723534 |
| 10 | 0.587685090 | 0.933076929 | 0.677595628 | 0.470335539 |
| 11 | 0.580286223 | 0.574296041 | 0.765027322 | 0.544365023 |
| 12 | 0.570550696 | 0.551511650 | 0.781420765 | 0.530463469 |
| 13 | 0.567472349 | 0.593119827 | 0.748633880 | 0.497880804 |
| 14 | 0.555082614 | 0.523499645 | 0.773224044 | 0.519191114 |
| 15 | 0.563568378 | 0.688962210 | 0.762295082 | 0.462472438 |
| 16 | 0.544781691 | 0.680713828 | 0.743169399 | 0.487682779 |
| 17 | 0.542021387 | 0.524355710 | 0.797814208 | 0.528008724 |
| 18 | 0.543138524 | 0.576771899 | 0.775956284 | 0.522491771 |
| 19 | 0.547189267 | 0.557074432 | 0.775956284 | 0.549714994 |
| 20 | 0.536623712 | 0.535596815 | 0.786885246 | 0.518318740 |
| 21 | 0.534194704 | 0.600645437 | 0.773224044 | 0.509233206 |
| 22 | 0.528354673 | 0.561373336 | 0.781420765 | 0.520807076 |
| 23 | 0.519792785 | 0.750138216 | 0.743169399 | 0.523447769 |
| 24 | 0.530390125 | 0.531259787 | 0.795081967 | 0.540560745 |
| 25 | 0.520408266 | 0.533677649 | 0.786885246 | 0.541939924 |
| 26 | 0.509141136 | 0.621848044 | 0.743169399 | 0.512653925 |
| 27 | 0.520468270 | 0.703933611 | 0.734972678 | 0.516590925 |
| 28 | 0.503277439 | 0.557746612 | 0.786885246 | 0.535816370 |
| 29 | 0.506234690 | 0.509310753 | 0.792349727 | 0.574164284 |
| 30 | 0.521770585 | 0.565339297 | 0.773224044 | 0.554193779 |
| 31 | 0.514067393 | 0.588310098 | 0.775956284 | 0.569670305 |
| 32 | 0.510180957 | 0.727209730 | 0.743169399 | 0.510222558 |
| 33 | 0.511709183 | 0.500314835 | 0.795081967 | 0.570551475 |
| 34 | 0.502240788 | 0.494233758 | 0.816939891 | 0.627795869 |
| 35 | 0.483254582 | 0.535192095 | 0.784153005 | 0.558741433 |
| 36 | 0.498668841 | 0.634401668 | 0.745901639 | 0.512124862 |
| 37 | 0.486557979 | 0.594720902 | 0.759562842 | 0.550658145 |
| 38 | 0.499145104 | 0.646454004 | 0.773224044 | 0.568636063 |
| 39 | 0.504070892 | 0.492847897 | 0.795081967 | 0.572078947 |
| 40 | 0.493350738 | 0.757847062 | 0.734972678 | 0.558392772 |
| 41 | 0.476284729 | 0.536750080 | 0.781420765 | 0.566998778 |
| 42 | 0.481239962 | 0.518582671 | 0.789617486 | 0.583715539 |
| 43 | 0.489051995 | 0.557170389 | 0.765027322 | 0.583415645 |
| 44 | 0.501609180 | 0.819657022 | 0.693989071 | 0.454130829 |
| 45 | 0.480984408 | 0.553349948 | 0.765027322 | 0.597148263 |
| 46 | 0.480316873 | 0.559845050 | 0.775956284 | 0.583627444 |
| 47 | 0.470253168 | 0.573137502 | 0.775956284 | 0.537753694 |
| 48 | 0.471040763 | 0.504722837 | 0.806010929 | 0.591188025 |
| 49 | 0.483883540 | 0.631558008 | 0.797814208 | 0.544276499 |
| 50 | 0.453801368 | 0.556580976 | 0.765027322 | 0.563873366 |
| 51 | 0.474408556 | 0.492588060 | 0.803278689 | 0.545949648 |
| 52 | 0.465985998 | 0.578497543 | 0.797814208 | 0.592734621 |
| 53 | 0.472004933 | 0.497290047 | 0.773224044 | 0.593068160 |
| 54 | 0.477338987 | 0.508591771 | 0.797814208 | 0.593941785 |
| 55 | 0.477039804 | 0.510912072 | 0.808743169 | 0.597652804 |
| 56 | 0.466405296 | 0.590859081 | 0.765027322 | 0.504036855 |
| 57 | 0.454527927 | 0.505033881 | 0.795081967 | 0.565629082 |
| 58 | 0.460897643 | 0.652293207 | 0.786885246 | 0.572484225 |
| 59 | 0.459729339 | 0.527525467 | 0.803278689 | 0.599103277 |
| 60 | 0.453754998 | 0.540316556 | 0.803278689 | 0.583656950 |
| 61 | 0.448232843 | 0.606097016 | 0.759562842 | 0.564527851 |
| 62 | 0.442065472 | 0.581881440 | 0.795081967 | 0.585390361 |
| 63 | 0.438213795 | 0.495877881 | 0.822404372 | 0.655097955 |
| 64 | 0.442223628 | 0.558645066 | 0.786885246 | 0.579633748 |
| 65 | 0.454576198 | 0.537183192 | 0.811475410 | 0.635367302 |
| 66 | 0.438006421 | 0.571833991 | 0.797814208 | 0.610762732 |
| 67 | 0.449755694 | 0.826412001 | 0.729508197 | 0.532136324 |
| 68 | 0.448087986 | 0.496987651 | 0.792349727 | 0.634000466 |
| **69** | 0.435552676 | 0.526093044 | 0.814207650 | 0.657127419 |
| 70 | 0.434326441 | 0.491634929 | 0.792349727 | 0.599609133 |
| 71 | 0.462585724 | 0.554533630 | 0.792349727 | 0.592320320 |
| 72 | 0.444371087 | 0.484275402 | 0.833333333 | 0.640452416 |
| 73 | 0.439278025 | 0.564530428 | 0.792349727 | 0.594134029 |
| 74 | 0.450125202 | 0.653251227 | 0.770491803 | 0.577039683 |
| 75 | 0.442007058 | 0.544299720 | 0.795081967 | 0.586771035 |
| 76 | 0.422450546 | 0.565657829 | 0.795081967 | 0.616220159 |
| 77 | 0.438597362 | 0.592946126 | 0.770491803 | 0.578968382 |
| 78 | 0.427155192 | 0.594666752 | 0.773224044 | 0.580450354 |
| 79 | 0.430052896 | 0.530776287 | 0.781420765 | 0.572134129 |
| 80 | 0.416941628 | 0.587610240 | 0.800546448 | 0.596479680 |

![Kurva historis V2 APTOS](../../reproduction/report_figures/historical_v2_aptos_training_curves.png)

### DDR V2 BG10 — 80 epoch validation

Epoch 1–59 menggunakan AMP FP16, 60–63 weighted FP32, dan 64–80 weighted FP32 dengan penghitungan norm gradien FP64. Perubahan objective membatasi pembacaan loss lintas fase.

| Epoch | Train Loss | Validation Loss | Validation Accuracy | Validation Macro-F1 |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 0.953021044 | 0.951698084 | 0.630990415 | 0.258752928 |
| 2 | 0.864125028 | 3.308556723 | 0.146166134 | 0.087032307 |
| 3 | 0.832094949 | 1.114721071 | 0.595047923 | 0.282922661 |
| 4 | 0.798618496 | 0.792308592 | 0.713258786 | 0.404000828 |
| 5 | 0.783135869 | 0.862767796 | 0.687699681 | 0.366149766 |
| 6 | 0.765560087 | 0.858204801 | 0.703674121 | 0.405832237 |
| 7 | 0.754513495 | 0.717358851 | 0.723642173 | 0.408448041 |
| 8 | 0.722542532 | 1.712558075 | 0.607028754 | 0.248770821 |
| 9 | 0.716422822 | 0.745650478 | 0.734025559 | 0.407240485 |
| 10 | 0.704981674 | 0.793670339 | 0.699680511 | 0.418162249 |
| 11 | 0.690039858 | 1.015393095 | 0.651757188 | 0.343032483 |
| 12 | 0.666049375 | 1.645217473 | 0.599840256 | 0.278314741 |
| 13 | 0.656335226 | 0.703892940 | 0.738019169 | 0.475394252 |
| 14 | 0.648696370 | 0.933566013 | 0.702875399 | 0.453322594 |
| 15 | 0.631653773 | 0.649795414 | 0.763578275 | 0.475733675 |
| 16 | 0.630098815 | 0.779455713 | 0.682907348 | 0.498378002 |
| 17 | 0.627675979 | 0.685959505 | 0.734025559 | 0.470637710 |
| 18 | 0.612234362 | 0.688300992 | 0.757188498 | 0.528692317 |
| 19 | 0.616321242 | 0.627434940 | 0.777156550 | 0.488897384 |
| 20 | 0.596970732 | 0.597461976 | 0.789936102 | 0.527269709 |
| 21 | 0.590332817 | 0.625374986 | 0.770766773 | 0.536299179 |
| 22 | 0.597096786 | 0.809269188 | 0.735623003 | 0.466833237 |
| 23 | 0.590420590 | 0.731254359 | 0.760383387 | 0.488142962 |
| 24 | 0.587719832 | 0.704040267 | 0.742811502 | 0.569135519 |
| 25 | 0.589852149 | 0.617200639 | 0.776357827 | 0.545124346 |
| 26 | 0.585576411 | 0.845162283 | 0.730031949 | 0.423256702 |
| 27 | 0.595741849 | 0.649696288 | 0.778753994 | 0.481220107 |
| 28 | 0.592505384 | 0.717083941 | 0.772364217 | 0.435650717 |
| 29 | 0.585092780 | 0.648401531 | 0.783546326 | 0.475058866 |
| 30 | 0.578941740 | 0.626489522 | 0.773162939 | 0.500904876 |
| 31 | 0.575985742 | 0.592626014 | 0.801118211 | 0.552221451 |
| 32 | 0.574745475 | 0.571338291 | 0.812300319 | 0.578945149 |
| 33 | 0.562716009 | 0.681227233 | 0.774760383 | 0.458816118 |
| 34 | 0.559321554 | 0.665444793 | 0.770766773 | 0.515661168 |
| 35 | 0.556057978 | 0.555571599 | 0.813897764 | 0.577074974 |
| 36 | 0.549256400 | 0.691877727 | 0.774760383 | 0.546181247 |
| 37 | 0.540840571 | 0.569003627 | 0.807507987 | 0.522961843 |
| 38 | 0.540689148 | 0.702560047 | 0.769169329 | 0.517477412 |
| 39 | 0.528297590 | 0.593974391 | 0.800319489 | 0.542606504 |
| 40 | 0.531648101 | 0.542322194 | 0.821086262 | 0.594843810 |
| 41 | 0.525392481 | 2.082735723 | 0.650159744 | 0.384690361 |
| 42 | 0.516821600 | 0.578464254 | 0.805111821 | 0.513806414 |
| 43 | 0.518535245 | 0.546640315 | 0.814696486 | 0.605874048 |
| 44 | 0.512860199 | 0.542930185 | 0.808306709 | 0.578547513 |
| 45 | 0.498055666 | 0.534233562 | 0.820287540 | 0.603395096 |
| 46 | 0.496887462 | 0.528787113 | 0.821884984 | 0.560596050 |
| 47 | 0.504630656 | 0.846999297 | 0.775559105 | 0.525075362 |
| 48 | 0.486234754 | 0.564259526 | 0.817092652 | 0.603744443 |
| 49 | 0.485910492 | 0.543931441 | 0.827476038 | 0.606662244 |
| 50 | 0.485248007 | 0.548311656 | 0.807507987 | 0.623805758 |
| 51 | 0.473122820 | 0.517052675 | 0.821884984 | 0.587481148 |
| 52 | 0.481732558 | 0.494440684 | 0.833865815 | 0.596042810 |
| 53 | 0.477671424 | 0.607126298 | 0.805111821 | 0.589107256 |
| 54 | 0.473157395 | 0.512848769 | 0.832268371 | 0.595827634 |
| 55 | 0.470775999 | 0.552495802 | 0.825079872 | 0.597272800 |
| 56 | 0.472058837 | 0.774911473 | 0.769169329 | 0.516366125 |
| 57 | 0.463413309 | 0.528193235 | 0.829872204 | 0.570160197 |
| 58 | 0.457308368 | 5.878699882 | 0.095846645 | 0.068298751 |
| 59 | 0.471362580 | 0.513880100 | 0.825079872 | 0.632503218 |
| 60 | 0.619204065 | 0.635587701 | 0.815495208 | 0.607562326 |
| 61 | 0.635347662 | 0.644368656 | 0.813099042 | 0.610678288 |
| 62 | 0.628927816 | 0.715479522 | 0.801916933 | 0.626961226 |
| 63 | 0.621462416 | 0.629044360 | 0.822683706 | 0.633206180 |
| 64 | 0.626616704 | 0.663693131 | 0.803514377 | 0.636910860 |
| 65 | 0.620806983 | 0.725123653 | 0.760383387 | 0.619410848 |
| 66 | 0.623419332 | 0.641742871 | 0.814696486 | 0.663704977 |
| 67 | 0.597046560 | 0.760380890 | 0.803514377 | 0.628589010 |
| 68 | 0.604773456 | 0.618092961 | 0.833067093 | 0.655189663 |
| 69 | 0.604240630 | 0.720752595 | 0.801118211 | 0.580534535 |
| 70 | 0.586110033 | 0.620512671 | 0.833067093 | 0.671788657 |
| 71 | 0.589423267 | 0.614167848 | 0.837859425 | 0.655984744 |
| **72** | 0.572472465 | 0.567171804 | 0.829872204 | 0.690323426 |
| 73 | 0.573540899 | 0.574356349 | 0.831469649 | 0.656161553 |
| 74 | 0.562714350 | 0.630887725 | 0.822683706 | 0.648229298 |
| 75 | 0.574923626 | 0.596541643 | 0.839456869 | 0.656919449 |
| 76 | 0.553652966 | 0.590208902 | 0.834664537 | 0.677854463 |
| 77 | 0.547899609 | 0.638863087 | 0.798722045 | 0.659940500 |
| 78 | 0.558448565 | 0.626060768 | 0.815495208 | 0.672755007 |
| 79 | 0.553967510 | 0.625284600 | 0.808306709 | 0.653064939 |
| 80 | 0.551709366 | 0.667835487 | 0.813099042 | 0.637946978 |

![Kurva historis V2 DDR](../../reproduction/report_figures/historical_v2_ddr_training_curves.png)

## Final test V2 80 epoch — hasil JSON berhash terverifikasi

| Dataset/run | Checkpoint dari validation | Best validation macro-F1 | N test | Accuracy test | Macro-F1 test | Macro specificity test |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| APTOS `aptos_v2_bg10_stable_80ep_seed42` | 69 | 0,6571274188819457 | 366 | 0,7896174863387978 | 0,6049524682606406 | 0,943038529998469 |
| DDR `ddr_v2_fp32_weighted_fp64clip_seed42` | 72 | 0,6903234255707131 | 1.252 | 0,8043130990415336 | 0,6202869401176525 | 0,9418616477896906 |

[APTOS final-test JSON](../../results/APTOS2019/v2_bg10_historical_80epochs/final_test_v2_80ep.json) SHA-256 `5db8e70f6c8ef78d0f9047f08f6ba1bcdae4c97c6d6b39c33ba58e8792b28e3c`; [DDR final-test JSON](../../results/DDR/v2_bg10_historical_80epochs/final_test_v2_80ep.json) SHA-256 `95b0a47ef36626e23e6721ebb1bfeacd6265a39939457ff37ae5115ca58381d6`. Keduanya cocok dengan hash yang dicatat peneliti dari server pada 2026-11-13. [Audit provenance](../../docs/V2_FINAL_TEST_PROVENANCE.md) menjelaskan bahwa identitas byte JSON tidak membuktikan lineage checkpoint, manifest, atau proses eksekusi.

### V2 BG10 APTOS — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 177 | 2 | 0 | 0 | 1 | 180 |
| 1 | 8 | 22 | 7 | 0 | 1 | 38 |
| 2 | 7 | 16 | 69 | 4 | 4 | 100 |
| 3 | 1 | 2 | 5 | 3 | 8 | 19 |
| 4 | 1 | 1 | 6 | 3 | 18 | 29 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 180 | 0.912371 | 0.983333 | 0.946524 | 0.908602 |
| 1 | 38 | 0.511628 | 0.578947 | 0.543210 | 0.935976 |
| 2 | 100 | 0.793103 | 0.690000 | 0.737968 | 0.932331 |
| 3 | 19 | 0.300000 | 0.157895 | 0.206897 | 0.979827 |
| 4 | 29 | 0.562500 | 0.620690 | 0.590164 | 0.958457 |

### V2 BG10 DDR — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 581 | 17 | 25 | 0 | 3 | 626 |
| 1 | 24 | 20 | 19 | 0 | 0 | 63 |
| 2 | 51 | 38 | 323 | 21 | 15 | 448 |
| 3 | 0 | 1 | 10 | 10 | 2 | 23 |
| 4 | 1 | 1 | 14 | 3 | 73 | 92 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 626 | 0.884323 | 0.928115 | 0.905690 | 0.878594 |
| 1 | 63 | 0.259740 | 0.317460 | 0.285714 | 0.952061 |
| 2 | 448 | 0.826087 | 0.720982 | 0.769964 | 0.915423 |
| 3 | 23 | 0.294118 | 0.434783 | 0.350877 | 0.980472 |
| 4 | 92 | 0.784946 | 0.793478 | 0.789189 | 0.982759 |

## Interpretasi dan kelanjutan

V2 BG10 80 epoch memiliki accuracy final test yang lebih tinggi daripada V1 pada kedua dataset dalam artefak yang tersedia. Perubahan preprocessing, training, dan pada DDR recovery terjadi bersama; data ini tidak mengisolasi penyebab peningkatan. Grade 1/3 DDR tetap lemah pada matriks V2, dengan support grade 3 hanya 23. Skor V2 tidak mewakili protokol homogen DDR. Perbandingan dengan paper dibatasi split, arsitektur kandidat, dan agregasi F1 paper yang tidak pasti. Karena itu eksperimen balancing berikutnya dipisahkan sebagai generasi ketiga, bukan penggantian hasil V2. Lanjut ke [eksperimen 3](03_experiment_balanced_training_and_ddr_recovery.ipynb).